In [17]:
from collections import defaultdict
from collections import deque
import timeit

In [18]:
input_filename = "day4_input.txt"

In [19]:
def part1_v1(input_filename):

    with open(input_filename, 'r') as f:
        diagram = f.read().splitlines()

    pos = {}
    xmax = len(diagram[0])
    ymax = len(diagram)
    for i in range(xmax):
        for j in range(ymax):
            pos[(i,j)] = diagram[ymax-1-j][i]

    for i in range(-1, xmax+1):
        pos[(i, ymax)] = '.'
        pos[(i, -1)] = '.'

    for j in range(ymax):
        pos[(xmax, j)] = '.'
        pos[(-1, j)] = '.'

    def count_paper(x, y, pos):
        paper = 0
        for i in [x-1, x, x+1]:
            for j in [y-1, y, y+1]:
                if (i!=x) or (j!=y):
                    if pos[(i,j)] == '@':
                        paper+=1
        return paper
    

    accessible_roles = 0

    test_pos = set()
    for i in range(xmax):
        for j in range(ymax):
            if pos[(i,j)] == '@':
                if count_paper(i, j, pos) < 4:
                    accessible_roles+=1

                    test_pos.add((i,j))

    return accessible_roles

part1_v1(input_filename)

1370

In [20]:
def part1_v2(input_filename):

    with open(input_filename, 'r') as f:
        diagram = f.read().splitlines()

    directions = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
    count = 0

    #Parsing and base case in one
    xmax = len(diagram)
    ymax = len(diagram[0])
    for i in range(xmax):
        for j in range(ymax):
            paper = (i, j)

            if diagram[i][j] == '@':
                paper_count = 0

                for (dx, dy) in directions:

                    prop = (i+dx, j+dy)
                    if (0<=prop[0]) and (prop[0]<xmax) and (0<=prop[1]) and (prop[1]<ymax):
                        if diagram[prop[0]][prop[1]] == '@':
                            paper_count += 1
                    
                if paper_count < 4:
                    count+=1
    return count

part1_v2(input_filename)

1370

In [21]:
iters = 10*4
execution_time = timeit.timeit(lambda: part1_v1(input_filename), number=iters)
print(f"Average Execution time version 1: {execution_time/iters} seconds")
execution_time = timeit.timeit(lambda: part1_v2(input_filename), number=iters)
print(f"Average Execution time version 2: {execution_time/iters} seconds")

Average Execution time version 1: 0.008320848975199625 seconds
Average Execution time version 2: 0.007954900000186171 seconds


# part 2

In [22]:
#more parsimonious solutions
def part2_v2(input_filename):
    with open(input_filename, 'r') as f:
        diagram = f.read().splitlines()
        xmax, ymax = len(diagram), len(diagram[0])

    #parse
    adj_tree = {
        (i,j): set(((i+dx,j+dy) 
            for (dx, dy) in [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)] 
            if (0 <= i+dx < xmax) and 0 <= j+dy < ymax 
            and diagram[i+dx][j+dy]=='@')) 
                for i in range(xmax) 
                for j in range(ymax)
                if diagram[i][j]=='@'
    }

    #initialse
    node_queue = deque()
    for paper, adj in adj_tree.items():
        if len(adj)<4:
            node_queue.append(paper)
            
    #clear
    counter=0
    while node_queue:
        paper = node_queue.popleft()
        adj_set = adj_tree.pop(paper)
        counter+=1
        
        for adj in adj_set:
            adj_tree[adj].remove(paper)

            if (len(adj_tree[adj])<4) and adj not in node_queue:
                node_queue.append(adj)

    return counter

In [23]:
# Faster Solution
def part2_v1(input_filename):
    
    with open(input_filename, 'r') as f:
        diagram = f.read().splitlines()

    directions = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
    adj = defaultdict(set)
    counts = defaultdict(int)
    clearable = set()
    clear_queue = deque()

    #Parsing and base case in one
    xmax = len(diagram)
    ymax = len(diagram[0])
    for i in range(xmax):
        for j in range(ymax):
            paper = (i, j)

            if diagram[i][j] == '@':
                counts[paper] = 0

                for (dx, dy) in directions:

                    prop = (i+dx, j+dy)
                    if (0<=prop[0]) and (prop[0]<xmax) and (0<=prop[1]) and (prop[1]<ymax):
                        if diagram[prop[0]][prop[1]] == '@':
                            adj[paper].add(prop)
                            counts[paper] += 1
                    
                if counts[paper] < 4:
                    clear_queue.append(paper)
                    clearable.add(paper)

    # Main search
    while clear_queue:
        paper = clear_queue.popleft()

        #clear by updating nearby
        for nearby_paper in adj[paper]:
            counts[nearby_paper] += -1

            if (counts[nearby_paper]<4) and (nearby_paper not in clearable):
                clear_queue.append(nearby_paper)
                clearable.add(nearby_paper)

    return len(clearable)



In [24]:
iters = 10*4
execution_time = timeit.timeit(lambda: part2_v1(input_filename), number=iters)
print(f"Average Execution time version 1: {execution_time/iters} seconds")
execution_time = timeit.timeit(lambda: part2_v2(input_filename), number=iters)
print(f"Average Execution time version 2: {execution_time/iters} seconds")

Average Execution time version 1: 0.021432819799883873 seconds
Average Execution time version 2: 0.0506271093749092 seconds
